In [ ]:
import pandas as pd
import numpy as np
import os

file_paths = [
    "CCLE_DATA/CCLE_BRCA_Genetic_features_0.csv",
    "CCLE_DATA/CCLE_BRCA_Genetic_features_1.csv", 
    "CCLE_DATA/CCLE_RPPA_Protein.csv",
    "CCLE_DATA/CCLE_RNAseq_genes_counts.gct",
    "CCLE_DATA/CCLE_RNAseq_genes_rpkm.gct",
    "CCLE_DATA/CCLE_RNAseq_rsem_genes_tpm.txt",
    "CCLE_DATA/CCLE_cnv_matrix.txt" 
]

def load_target_features(contributed_omic_csv):
    """Load the target features from multi_omic_contribution_analysis.csv"""
    print("\n Loading target features from contribution analysis...")
    
    try:
        contrib_df = pd.read_csv(contributed_omic_csv)
        
        # Group features by omic type
        target_features = {
            'gene': set(),
            'protein': set(),
            'cnv': set(),
            'mutation': set()
        }
        
        for _, row in contrib_df.iterrows():
            omic_type = row['omic_type']
            feature_name = row['feature_name']
            
            if omic_type in target_features:
                target_features[omic_type].add(feature_name)
        
        print(f"  Loaded EXACT target features:")
        print(f"   Gene features: {len(target_features['gene'])}")
        print(f"   Protein features: {len(target_features['protein'])}")
        print(f"   CNV features: {len(target_features['cnv'])}")
        print(f"   Mutation features: {len(target_features['mutation'])}")
        
        return target_features
        
    except FileNotFoundError:
        print("\n multi_omic_contribution_analysis.csv not found!")
        print("   Please ensure the file exists in 'omic_contribution_analysis/' directory")
        return None
    except Exception as e:
        print(f"\n Error loading contribution analysis: {e}")
        return None

def is_feature_target(feature_name, omic_type, target_features):
    """Check if a feature is in our target list - STRICT EXACT MATCHING with mutation suffix handling"""
    if not target_features or omic_type not in target_features:
        return False
    
    feature_name_clean = str(feature_name).strip()
    
    # For mutation features, remove common suffixes like _mut, _mutation etc.
    if omic_type == 'mutation':
        # Remove mutation-specific suffixes to get the base gene name
        mutation_suffixes = ['_mut', '_mutation', '_del', '_ins', '_snv', '_indel']
        feature_base = feature_name_clean
        
        for suffix in mutation_suffixes:
            if feature_base.lower().endswith(suffix.lower()):
                feature_base = feature_base[:-len(suffix)]
                break
        
        # Compare the base name (without suffix) with target features
        for target in target_features[omic_type]:
            target_clean = str(target).strip()
            
            # Exact match (case-insensitive) using base name
            if feature_base.lower() == target_clean.lower():
                return True
    
    else:
        # For non-mutation features, do exact matches (case-insensitive)
        for target in target_features[omic_type]:
            target_clean = str(target).strip()
            
            # Exact match (case-insensitive)
            if feature_name_clean.lower() == target_clean.lower():
                return True
            
            # For genes only: handle common variations like version numbers
            if omic_type == 'gene':
                # Remove version numbers for gene IDs (e.g., ENSG00000123.1 -> ENSG00000123)
                feature_base = feature_name_clean.split('.')[0]
                target_base = target_clean.split('.')[0]
                
                if feature_base.lower() == target_base.lower():
                    return True
    
    return False

def clean_cell_line_name(name):
    """Clean cell line name by removing tissue suffixes and special characters"""
    if '_' in name:
        return name.split('_')[0]
    return name.replace('-', '').replace(' ', '')

def create_gene_symbol_mapping():
    """Create Ensembl ID to Gene Symbol mapping from GCT file"""
    print("\n Creating Ensembl ID → Gene Symbol mapping...")
    
    gct_df = pd.read_csv(file_paths[3], sep='\t', skiprows=2)
    
    gene_mapping = {}
    for _, row in gct_df.iterrows():
        ensembl_id = row['Name']  # ENSG00000223972.4
        gene_symbol = row['Description']  # DDX11L1
        
        # Store both with and without version number
        gene_mapping[ensembl_id] = gene_symbol
        if '.' in ensembl_id:
            clean_id = ensembl_id.split('.')[0]
            gene_mapping[clean_id] = gene_symbol
    
    print(f"  Created mapping for {len(gene_mapping)} Ensembl IDs")
    return gene_mapping

def extract_table_1_genetic_features(target_features):
    """Extract Table 1: Genetic Features (Mutations) - FILTERED"""
    print(f"\n Table 1: Genetic Features (Mutations) - Filtered")
    print("-" * 60)
    
    df = pd.read_csv(file_paths[0])
    breast_df = df[df['GDSC Desc1'] == 'breast']
    mutated_features = breast_df[breast_df['IS Mutated'] == 1]
    
    data_list = []
    total_features = 0
    filtered_features = 0
    
    for _, row in mutated_features.iterrows():
        total_features += 1
        genetic_feature = row['Genetic Feature']
        
        # Check if this feature is in our target list
        if is_feature_target(genetic_feature, 'mutation', target_features):
            filtered_features += 1
            data_list.append({
                'Cell_Line': clean_cell_line_name(row['Cell Line Name']),
                'Cell_Line_Original': row['Cell Line Name'],
                'COSMIC_ID': row['COSMIC ID'],
                'Genetic_Feature': genetic_feature,
                'IS_Mutated': row['IS Mutated'],
                'Data_Type': 'Mutation',
                'Source_Table': 'Table_1'
            })
    
    result_df = pd.DataFrame(data_list)
    
    print(f" Filtered {filtered_features}/{total_features} mutation features")
    print(f" Found {len(result_df)} mappings across {result_df['Cell_Line'].nunique() if len(result_df) > 0 else 0} cell lines")
    
    if len(result_df) > 0:
        print(" \nSample:")
        print(result_df.head()[['Cell_Line', 'Genetic_Feature']].to_string(index=False))
    
    return result_df

def extract_table_2_genetic_features(target_features):
    """Extract Table 2: Genetic Features (Mutations) - FILTERED"""
    print(f"\n Table 2: Genetic Features (Mutations) - Filtered")
    print("-" * 60)
    
    df = pd.read_csv(file_paths[1])
    breast_df = df[df['GDSC Desc1'] == 'breast']
    mutated_features = breast_df[breast_df['IS Mutated'] == 1]
    
    data_list = []
    total_features = 0
    filtered_features = 0
    
    for _, row in mutated_features.iterrows():
        total_features += 1
        genetic_feature = row['Genetic Feature']
        
        # Check if this feature is in our target list
        if is_feature_target(genetic_feature, 'mutation', target_features):
            filtered_features += 1
            data_list.append({
                'Cell_Line': clean_cell_line_name(row['Cell Line Name']),
                'Cell_Line_Original': row['Cell Line Name'],
                'COSMIC_ID': row['COSMIC ID'],
                'Genetic_Feature': genetic_feature,
                'IS_Mutated': row['IS Mutated'],
                'Data_Type': 'Mutation',
                'Source_Table': 'Table_2'
            })
    
    result_df = pd.DataFrame(data_list)
    
    print(f" Filtered {filtered_features}/{total_features} mutation features")
    print(f" Found {len(result_df)} mappings across {result_df['Cell_Line'].nunique() if len(result_df) > 0 else 0} cell lines")
    
    if len(result_df) > 0:
        print(" \nSample:")
        print(result_df.head()[['Cell_Line', 'Genetic_Feature']].to_string(index=False))
    
    return result_df

def extract_table_3_protein_expression(target_features):
    """Extract Table 3: Protein Expression - FILTERED"""
    print(f"\n Table 3: Protein Expression - Filtered")
    print("-" * 60)
    
    df = pd.read_csv(file_paths[2])
    
    # Find breast cancer cell lines by keywords
    breast_keywords = ['MCF7', 'MDA', 'BT474', 'BT549', 'SKBR3', 'T47D', 'AU565', 
                      'HCC', 'CAL51', 'CAMA1', 'EFM', 'JIMT1', 'KPL1', 'UACC', 'ZR']
    
    breast_mask = df['Cell Line Name'].str.contains('|'.join(breast_keywords), case=False, na=False)
    breast_df = df[breast_mask]
    
    protein_cols = [col for col in df.columns if col != 'Cell Line Name']
    
    data_list = []
    total_features = 0
    filtered_features = 0
    
    for _, row in breast_df.iterrows():
        cell_line = clean_cell_line_name(row['Cell Line Name'])
        
        for protein in protein_cols:
            total_features += 1
            value = row[protein]
            
            # Check if this protein is in our target list AND has positive expression
            if (pd.notna(value) and value > 0 and 
                is_feature_target(protein, 'protein', target_features)):
                
                filtered_features += 1
                data_list.append({
                    'Cell_Line': cell_line,
                    'Cell_Line_Original': row['Cell Line Name'],
                    'Protein_Name': protein,
                    'Expression_Value': value,
                    'Data_Type': 'Protein',
                    'Source_Table': 'Table_3'
                })
    
    result_df = pd.DataFrame(data_list)
    
    print(f" Filtered {filtered_features}/{total_features} protein features")
    print(f" Found {len(result_df)} mappings across {result_df['Cell_Line'].nunique() if len(result_df) > 0 else 0} cell lines")
    
    if len(result_df) > 0:
        print(" \nSample:")
        print(result_df.head()[['Cell_Line', 'Protein_Name', 'Expression_Value']].to_string(index=False))
    
    return result_df

def extract_table_4_gene_expression(target_features):
    """Extract Table 4: Gene Expression Counts - FILTERED"""
    print(f"\n Table 4: Gene Expression Counts - Filtered")
    print("-" * 60)
    
    df = pd.read_csv(file_paths[3], sep='\t', skiprows=2)
    breast_columns = [col for col in df.columns if '_BREAST' in col]
    
    data_list = []
    total_genes_checked = 0
    matched_genes = 0
    gene_matches = set()  # Track which genes actually matched
    
    for _, row in df.iterrows():
        ensembl_id = row['Name']
        gene_symbol = row['Description']
        total_genes_checked += 1
        
        # Check if this gene is in our target list (EXACT MATCH ONLY)
        if (is_feature_target(gene_symbol, 'gene', target_features) or 
            is_feature_target(ensembl_id, 'gene', target_features)):
            
            matched_genes += 1
            gene_matches.add(gene_symbol)  # Track the matched gene
            
            for col in breast_columns:
                value = row[col]
                if pd.notna(value) and value > 0:
                    cell_line = clean_cell_line_name(col)
                    data_list.append({
                        'Cell_Line': cell_line,
                        'Cell_Line_Original': col,
                        'Gene_ID': ensembl_id,
                        'Gene_Symbol': gene_symbol,
                        'Expression_Value': value,
                        'Data_Type': 'Gene_Counts',
                        'Source_Table': 'Table_4'
                    })
    
    result_df = pd.DataFrame(data_list)
    
    print(f" EXACT MATCHING RESULTS:")
    print(f"   Total genes in data: {total_genes_checked}")
    print(f"   Genes matched to targets: {matched_genes}")
    print(f"   Final expression mappings: {len(result_df)}")
    print(f"   Cell lines: {result_df['Cell_Line'].nunique() if len(result_df) > 0 else 0}")
    
    if len(result_df) > 0:
        print("\n Sample:")
        print(result_df.head()[['Cell_Line', 'Gene_Symbol', 'Expression_Value']].to_string(index=False))
    
    return result_df

def extract_table_5_gene_expression(target_features):
    """Extract Table 5: Gene Expression RPKM - FILTERED"""
    print(f"\n Table 5: Gene Expression RPKM - Filtered")
    print("-" * 60)
    
    df = pd.read_csv(file_paths[4], sep='\t', skiprows=2)
    breast_columns = [col for col in df.columns if '_BREAST' in col]
    
    data_list = []
    total_features = 0
    filtered_features = 0
    
    for _, row in df.iterrows():
        ensembl_id = row['Name']
        gene_symbol = row['Description']
        
        # Check if this gene is in our target list
        if (is_feature_target(gene_symbol, 'gene', target_features) or 
            is_feature_target(ensembl_id, 'gene', target_features)):
            
            for col in breast_columns:
                total_features += 1
                value = row[col]
                if pd.notna(value) and value > 0:
                    filtered_features += 1
                    cell_line = clean_cell_line_name(col)
                    data_list.append({
                        'Cell_Line': cell_line,
                        'Cell_Line_Original': col,
                        'Gene_ID': ensembl_id,
                        'Gene_Symbol': gene_symbol,
                        'Expression_Value': value,
                        'Data_Type': 'Gene_RPKM',
                        'Source_Table': 'Table_5'
                    })
    
    result_df = pd.DataFrame(data_list)
    
    print(f" Found {len(result_df)} filtered gene expression mappings")
    print(f" Across {result_df['Cell_Line'].nunique() if len(result_df) > 0 else 0} cell lines")
    
    if len(result_df) > 0:
        print(" \nSample:")
        print(result_df.head()[['Cell_Line', 'Gene_Symbol', 'Expression_Value']].to_string(index=False))
    
    return result_df

def extract_table_6_gene_tpm(gene_mapping, target_features):
    """Extract Table 6: Gene TPM Expression - FILTERED"""
    print(f"\n Table 6: Gene TPM Expression - Filtered")
    print("-" * 60)
    
    # Read header to find breast cancer columns
    with open(file_paths[5], 'r') as f:
        header_line = f.readline().strip()
    
    columns = header_line.split('\t')
    if columns[0].startswith('1: '):
        columns[0] = columns[0][3:]
    
    breast_columns = [col for col in columns if '_BREAST' in col]
    breast_indices = [columns.index(col) for col in breast_columns]
    
    print(f" Found {len(breast_columns)} breast cancer cell lines")
    
    data_list = []
    total_features = 0
    filtered_features = 0
    
    with open(file_paths[5], 'r') as f:
        header = f.readline()  # Skip header
        
        for line_count, line in enumerate(f):
            values = line.strip().split('\t')
            if len(values) <= max(breast_indices):
                continue
                
            gene_id = values[0]
            transcript_ids = values[1] if len(values) > 1 else ""
            
            # Map to gene symbol using our mapping
            gene_symbol = gene_mapping.get(gene_id, gene_id)
            if gene_symbol == gene_id and '.' in gene_id:
                clean_id = gene_id.split('.')[0]
                gene_symbol = gene_mapping.get(clean_id, gene_id)
            
            # Check if this gene is in our target list
            if (is_feature_target(gene_symbol, 'gene', target_features) or 
                is_feature_target(gene_id, 'gene', target_features)):
                
                # Extract values for each breast cancer cell line
                for i, col_idx in enumerate(breast_indices):
                    total_features += 1
                    try:
                        value = float(values[col_idx])
                        if value > 0:
                            filtered_features += 1
                            cell_line = clean_cell_line_name(breast_columns[i])
                            data_list.append({
                                'Cell_Line': cell_line,
                                'Cell_Line_Original': breast_columns[i],
                                'Gene_ID': gene_id,
                                'Gene_Symbol': gene_symbol,
                                'Transcript_IDs': transcript_ids,
                                'TPM_Value': value,
                                'Data_Type': 'Gene_TPM',
                                'Source_Table': 'Table_6'
                            })
                    except (ValueError, IndexError):
                        continue
    
    result_df = pd.DataFrame(data_list)
    
    if len(result_df) > 0:
        unique_cells = result_df['Cell_Line'].nunique()
        unique_genes = result_df['Gene_Symbol'].nunique()
        
        print(f" Found {len(result_df)} filtered TPM expressions")
        print(f" {unique_cells} cell lines × {unique_genes} unique genes")
        
        print(" \nSample:")
        print(result_df.head()[['Cell_Line', 'Gene_Symbol', 'TPM_Value']].to_string(index=False))
    
    return result_df

def extract_table_7_cnv_features(target_features):
    """Extract Table 7: CNV Features - FILTERED"""
    print(f"\n Table 7: CNV Features - Filtered")
    print("-" * 60)
    
    with open(file_paths[6], 'r') as f:
        lines = f.readlines()
    
    # Parse the header structure
    header_line = lines[0].strip().split('\t')
    cell_line_names = header_line[3:]
    
    tissue_line = lines[1].strip().split('\t')
    tissue_types = tissue_line[3:]
    
    # Find breast cancer cell lines by checking tissue type
    breast_cell_indices = []
    breast_cell_names = []
    
    for i, tissue in enumerate(tissue_types):
        if tissue == 'breast':
            breast_cell_indices.append(i)
            breast_cell_names.append(cell_line_names[i])
    
    print(f" Found {len(breast_cell_names)} breast cancer cell lines")
    
    data_list = []
    total_features = 0
    filtered_features = 0
    
    # Process each gene row
    for line_idx in range(3, len(lines)):
        line = lines[line_idx].strip()
        if not line:
            continue
            
        values = line.split('\t')
        if len(values) < 4:
            continue
            
        gene_symbol = values[0] if values[0] != 'na' else None
        gene_id = values[2] if len(values) > 2 and values[2] != 'na' else None
        
        cnv_feature = gene_symbol if gene_symbol and gene_symbol != 'na' else gene_id
        
        if not cnv_feature or cnv_feature == 'na':
            continue
        
        # Check if this CNV feature is in our target list
        if is_feature_target(cnv_feature, 'cnv', target_features):
            cnv_values = values[3:]
            
            for i, breast_idx in enumerate(breast_cell_indices):
                total_features += 1
                if breast_idx < len(cnv_values):
                    try:
                        cnv_value = float(cnv_values[breast_idx])
                        if cnv_value != 0.0:
                            filtered_features += 1
                            cell_line = clean_cell_line_name(breast_cell_names[i])
                            data_list.append({
                                'Cell_Line': cell_line,
                                'Cell_Line_Original': breast_cell_names[i],
                                'CNV_Feature': cnv_feature,
                                'CNV_Value': cnv_value,
                                'Data_Type': 'CNV',
                                'Source_Table': 'Table_7'
                            })
                    except (ValueError, IndexError):
                        continue
    
    result_df = pd.DataFrame(data_list)
    
    print(f" Filtered {filtered_features} CNV mappings from target features")
    print(f" Found {len(result_df)} mappings across {result_df['Cell_Line'].nunique() if len(result_df) > 0 else 0} cell lines")
    
    if len(result_df) > 0:
        print(" \nSample:")
        print(result_df.head()[['Cell_Line', 'CNV_Feature', 'CNV_Value']].to_string(index=False))
    
    return result_df

def save_all_results(results_dict, target_features):
    """Save all filtered results to organized CSV files"""
    print(f"\n Saving Filtered Multi-Omics Tables")
    print("=" * 60)
    
    output_dir = 'filtered_cell_line_data'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    saved_files = []
    summary_data = []
    
    for table_name, df in results_dict.items():
        if len(df) > 0:
            output_path = os.path.join(output_dir, f"{table_name}.csv")
            df.to_csv(output_path, index=False)
            saved_files.append(output_path)
            
            unique_cells = df['Cell_Line'].nunique()
            
            if 'Genetic_Feature' in df.columns:
                unique_features = df['Genetic_Feature'].nunique()
                feature_type = 'Genetic Features'
            elif 'Protein_Name' in df.columns:
                unique_features = df['Protein_Name'].nunique()
                feature_type = 'Proteins'
            elif 'Gene_Symbol' in df.columns:
                unique_features = df['Gene_Symbol'].nunique()
                feature_type = 'Genes'
            elif 'CNV_Feature' in df.columns:
                unique_features = df['CNV_Feature'].nunique()
                feature_type = 'CNV Features'
            else:
                unique_features = 0
                feature_type = 'Unknown'
            
            summary_data.append({
                'Table': table_name,
                'Feature_Type': feature_type,
                'Total_Mappings': len(df),
                'Unique_Cell_Lines': unique_cells,
                'Unique_Features': unique_features,
                'Source': df['Source_Table'].iloc[0] if 'Source_Table' in df.columns else 'Unknown'
            })
            
            print(f" Saved: {output_path} ({len(df)} mappings)")
    
    # Save summary report
    summary_df = pd.DataFrame(summary_data)
    summary_path = os.path.join(output_dir, 'Filtered_Summary_Report.csv')
    summary_df.to_csv(summary_path, index=False)
    print(f" Saved: {summary_path}")
    
    # Save target features used for filtering
    target_features_data = []
    for omic_type, features in target_features.items():
        for feature in features:
            target_features_data.append({
                'Omic_Type': omic_type,
                'Feature_Name': feature
            })
    
    target_df = pd.DataFrame(target_features_data)
    target_path = os.path.join(output_dir, 'Target_Features_Used.csv')
    target_df.to_csv(target_path, index=False)
    print(f" Saved: {target_path}")
    
    # Create master cell line list
    all_cell_lines = set()
    for df in results_dict.values():
        if len(df) > 0 and 'Cell_Line' in df.columns:
            all_cell_lines.update(df['Cell_Line'].unique())
    
    cell_line_df = pd.DataFrame({
        'Unique_Breast_Cancer_Cell_Lines': sorted(list(all_cell_lines))
    })
    cell_line_path = os.path.join(output_dir, 'Master_Cell_Line_List.csv')
    cell_line_df.to_csv(cell_line_path, index=False)
    print(f" Saved: {cell_line_path}")
    
    print(f"   All files saved in '{output_dir}/' directory")

def run_filtered_extraction(contributed_omic_csv="omic_contribution_analysis/multi_omic_contribution_analysis.csv"):
    """Run filtered multi-omics extraction based on contribution analysis"""
    print("Filtered Multi-Omic Data Extraction")
    print("=" * 80)

    # Load target features from contribution analysis
    target_features = load_target_features(contributed_omic_csv)
    if target_features is None:
        print("\n Cannot proceed without target features!")
        return None
    
    # Create gene mapping
    gene_mapping = create_gene_symbol_mapping()
    
    results = {}
    
    try:
        results['Table_1_Genetic_Features'] = extract_table_1_genetic_features(target_features)
        results['Table_2_Genetic_Features'] = extract_table_2_genetic_features(target_features)
        results['Table_3_Protein_Expression'] = extract_table_3_protein_expression(target_features)
        results['Table_4_Gene_Expression_Counts'] = extract_table_4_gene_expression(target_features)
        results['Table_5_Gene_Expression_RPKM'] = extract_table_5_gene_expression(target_features)
        results['Table_6_Gene_TPM_Expression'] = extract_table_6_gene_tpm(gene_mapping, target_features)
        results['Table_7_CNV_Features'] = extract_table_7_cnv_features(target_features)
        
        save_all_results(results, target_features)
        
        print(f"\n Filtered Extraction Complete!")
        
    except Exception as e:
        print(f"\n Error during extraction: {e}")
        import traceback
        traceback.print_exc()

In [2]:
import pandas as pd
import numpy as np
import os

def load_contribution_analysis():
    """Load the multi-omic contribution analysis data"""
    print(" \nLoading multi-omic contribution analysis...")
    
    contrib_df = pd.read_csv('omic_contribution_analysis/multi_omic_contribution_analysis.csv')
    
    print(f" Loaded {len(contrib_df)} contribution records")
    print(f" Cancer subtypes: {contrib_df['cancer_subtype'].unique()}")
    print(f" Omic types: {contrib_df['omic_type'].unique()}")
    
    return contrib_df

def load_extracted_tables(cell_line_table_dir):
    """Load all the extracted cell line tables INCLUDING 7th CNV table"""
    print("\n Loading extracted cell line tables...")
    
    table_dir = cell_line_table_dir
    tables = {}
    
    # Updated to include all 7 tables
    table_files = {
        'genetic_1': 'Table_1_All_Genetic_Features.csv',
        'genetic_2': 'Table_2_All_Genetic_Features.csv', 
        'protein': 'Table_3_All_Protein_Expression.csv',
        'gene_counts': 'Table_4_All_Gene_Expression_Counts.csv',
        'gene_rpkm': 'Table_5_All_Gene_Expression_RPKM.csv',
        'gene_tpm': 'Table_6_All_Gene_TPM_Expression.csv',
        'cnv_matrix': 'Table_7_All_CNV_Features.csv'
    }
    
    for table_key, filename in table_files.items():
        file_path = os.path.join(table_dir, filename)
        if os.path.exists(file_path):
            tables[table_key] = pd.read_csv(file_path)
            print(f"  Loaded {table_key}: {len(tables[table_key])} mappings")
        else:
            print(f"\n File not found: {file_path}")
            tables[table_key] = pd.DataFrame()
    
    return tables

def find_cell_lines_for_feature(feature_name, omic_type, tables):
    """Find cell lines that have a specific feature - UPDATED for 7th CNV table"""
    
    cell_lines = []
    
    if omic_type in ['mutation']:
        # Look in genetic features tables (Table 1 & 2) for mutations
        for table_key in ['genetic_1', 'genetic_2']:
            if table_key in tables and len(tables[table_key]) > 0:
                table_df = tables[table_key]
                
                if 'Genetic_Feature' in table_df.columns:
                    matching_rows = table_df[
                        table_df['Genetic_Feature'].str.contains(feature_name, case=False, na=False, regex=False)
                    ]
                    cell_lines.extend(matching_rows['Cell_Line'].tolist())
    
    elif omic_type == 'cnv':
        #  NEW: Look in CNV matrix table (Table 7) FIRST, then genetic tables
        if 'cnv_matrix' in tables and len(tables['cnv_matrix']) > 0:
            cnv_df = tables['cnv_matrix']
            
            # Search in CNV_Feature column
            if 'CNV_Feature' in cnv_df.columns:
                matching_rows = cnv_df[
                    cnv_df['CNV_Feature'].str.contains(feature_name, case=False, na=False, regex=False)
                ]
                cell_lines.extend(matching_rows['Cell_Line'].tolist())
                
                # Also try exact matches
                exact_matches = cnv_df[cnv_df['CNV_Feature'] == feature_name]
                cell_lines.extend(exact_matches['Cell_Line'].tolist())
        
        # Also check genetic features tables for CNV-related features
        for table_key in ['genetic_1', 'genetic_2']:
            if table_key in tables and len(tables[table_key]) > 0:
                table_df = tables[table_key]
                
                if 'Genetic_Feature' in table_df.columns:
                    # Look for CNV-related features
                    cnv_rows = table_df[
                        table_df['Genetic_Feature'].str.contains(feature_name, case=False, na=False, regex=False)
                    ]
                    cell_lines.extend(cnv_rows['Cell_Line'].tolist())
    
    elif omic_type == 'protein':
        # Look in protein expression table (Table 3)
        if 'protein' in tables and len(tables['protein']) > 0:
            table_df = tables['protein']
            if 'Protein_Name' in table_df.columns:
                matching_rows = table_df[
                    table_df['Protein_Name'].str.contains(feature_name, case=False, na=False, regex=False)
                ]
                cell_lines.extend(matching_rows['Cell_Line'].tolist())
    
    elif omic_type == 'gene':
        # Look in gene expression tables (Table 4, 5, 6)
        for table_key in ['gene_counts', 'gene_rpkm', 'gene_tpm']:
            if table_key in tables and len(tables[table_key]) > 0:
                table_df = tables[table_key]
                
                # Try different column names for gene identifiers
                gene_columns = []
                if 'Gene_Symbol' in table_df.columns:
                    gene_columns.append('Gene_Symbol')
                if 'Gene_Name' in table_df.columns:
                    gene_columns.append('Gene_Name')
                if 'Gene_ID' in table_df.columns:
                    gene_columns.append('Gene_ID')
                if 'Description' in table_df.columns:
                    gene_columns.append('Description')
                
                for gene_col in gene_columns:
                    matching_rows = table_df[
                        table_df[gene_col].str.contains(feature_name, case=False, na=False, regex=False)
                    ]
                    cell_lines.extend(matching_rows['Cell_Line'].tolist())
    
    # Remove duplicates and return
    unique_cell_lines = list(set(cell_lines))
    return unique_cell_lines

def create_enhanced_contribution_analysis(contrib_df, tables):

    print("\n Mapping features to cell lines...")
    
    enhanced_data = []
    feature_to_celllines = {}
    cnv_mapping_stats = {'total_cnv_features': 0, 'mapped_cnv_features': 0}
    
    total_features = len(contrib_df)
    
    for idx, row in contrib_df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{total_features} features...")
        
        feature_name = row['feature_name']
        omic_type = row['omic_type']
        
        # Track CNV features
        if omic_type == 'cnv':
            cnv_mapping_stats['total_cnv_features'] += 1
        
        # Find cell lines for this feature
        mapped_cell_lines = find_cell_lines_for_feature(feature_name, omic_type, tables)
        
        # Track successful CNV mappings
        if omic_type == 'cnv' and len(mapped_cell_lines) > 0:
            cnv_mapping_stats['mapped_cnv_features'] += 1
        
        # Store the mapping for matrix creation
        feature_key = f"{feature_name}_{omic_type}"
        feature_to_celllines[feature_key] = {
            'cell_lines': mapped_cell_lines,
            'omic_type': omic_type,
            'feature_name': feature_name
        }
        
        # Create enhanced record
        enhanced_record = {
            'cancer_subtype': row['cancer_subtype'],
            'omic_type': row['omic_type'],
            'rank': row['rank'],
            'feature_name': row['feature_name'],
            'feature_index': row['feature_index'],
            'importance_score': row['importance_score'],
            'percentile_rank': row['percentile_rank'],
            'significance_level': row['significance_level'],
            'attention_component': row['attention_component'],
            'metapath_component': row['metapath_component'],
            'raw_score': row['raw_score'],
            'method': row['method'],
            'mapped_cell_lines': '; '.join(mapped_cell_lines) if mapped_cell_lines else 'None',
            'cell_line_count': len(mapped_cell_lines),
            'has_cell_line_data': len(mapped_cell_lines) > 0
        }
        
        enhanced_data.append(enhanced_record)
    
    enhanced_df = pd.DataFrame(enhanced_data)
    
    # Report CNV mapping success
    cnv_success_rate = (cnv_mapping_stats['mapped_cnv_features'] / cnv_mapping_stats['total_cnv_features'] * 100) if cnv_mapping_stats['total_cnv_features'] > 0 else 0
    print(f"\n Enhanced {len(enhanced_df)} contribution records with cell line mappings")
    
    return enhanced_df, feature_to_celllines

def create_consistent_cell_line_matrix(tables, feature_to_celllines):
    """
    Create a CONSISTENT matrix using the same logic as feature mapping - UPDATED for CNV matrix
    """
    print("\n Creating Consistent Cell Line × Omic Data Matrix...")
    
    # Get all unique cell lines from all tables
    all_cell_lines = set()
    for table_key, table_df in tables.items():
        if len(table_df) > 0 and 'Cell_Line' in table_df.columns:
            all_cell_lines.update(table_df['Cell_Line'].unique())
    
    all_cell_lines = sorted(list(all_cell_lines))
    print(f" Found {len(all_cell_lines)} unique cell lines across all tables")
    
    # Create mapping of cell line to omic types using the SAME logic as feature mapping
    cell_line_omic_data = {}
    
    for cell_line in all_cell_lines:
        cell_line_omic_data[cell_line] = {
            'gene_features': set(),
            'protein_features': set(),
            'cnv_features': set(),
            'mutation_features': set()
        }
    
    # Use the feature mappings to populate the matrix consistently
    for feature_key, mapping_info in feature_to_celllines.items():
        omic_type = mapping_info['omic_type']
        feature_name = mapping_info['feature_name']
        cell_lines = mapping_info['cell_lines']
        
        for cell_line in cell_lines:
            if cell_line in cell_line_omic_data:
                if omic_type == 'gene':
                    cell_line_omic_data[cell_line]['gene_features'].add(feature_name)
                elif omic_type == 'protein':
                    cell_line_omic_data[cell_line]['protein_features'].add(feature_name)
                elif omic_type == 'cnv':
                    cell_line_omic_data[cell_line]['cnv_features'].add(feature_name)
                elif omic_type == 'mutation':
                    cell_line_omic_data[cell_line]['mutation_features'].add(feature_name)
    
    #  NEW: Add features found directly in CNV matrix table (Table 7)
    if 'cnv_matrix' in tables and len(tables['cnv_matrix']) > 0:
        cnv_df = tables['cnv_matrix']
        
        for cell_line in all_cell_lines:
            cell_cnv_data = cnv_df[cnv_df['Cell_Line'] == cell_line]
            
            for _, row in cell_cnv_data.iterrows():
                if 'CNV_Feature' in row and pd.notna(row['CNV_Feature']):
                    cnv_feature = str(row['CNV_Feature'])
                    cell_line_omic_data[cell_line]['cnv_features'].add(cnv_feature)
    
    # Also add features found directly in other tables
    for cell_line in all_cell_lines:
        # Gene features from expression tables
        for table_key in ['gene_counts', 'gene_rpkm', 'gene_tpm']:
            if table_key in tables and len(tables[table_key]) > 0:
                table_df = tables[table_key]
                cell_data = table_df[table_df['Cell_Line'] == cell_line]
                
                for _, row in cell_data.iterrows():
                    gene_name = None
                    if 'Gene_Symbol' in row:
                        gene_name = row['Gene_Symbol']
                    elif 'Gene_Name' in row:
                        gene_name = row['Gene_Name']
                    elif 'Description' in row:
                        gene_name = row['Description']
                    
                    if gene_name and pd.notna(gene_name):
                        cell_line_omic_data[cell_line]['gene_features'].add(str(gene_name))
        
        # Protein features
        if 'protein' in tables and len(tables['protein']) > 0:
            table_df = tables['protein']
            cell_data = table_df[table_df['Cell_Line'] == cell_line]
            for _, row in cell_data.iterrows():
                if 'Protein_Name' in row and pd.notna(row['Protein_Name']):
                    cell_line_omic_data[cell_line]['protein_features'].add(str(row['Protein_Name']))
        
        # Genetic features from Tables 1 & 2 (mutations and additional CNVs)
        for table_key in ['genetic_1', 'genetic_2']:
            if table_key in tables and len(tables[table_key]) > 0:
                table_df = tables[table_key]
                cell_data = table_df[table_df['Cell_Line'] == cell_line]
                
                for _, row in cell_data.iterrows():
                    if 'Genetic_Feature' in row and pd.notna(row['Genetic_Feature']):
                        genetic_feature = str(row['Genetic_Feature'])
                        
                        # Check if this feature was categorized in our contribution analysis
                        found_as_cnv = False
                        found_as_mutation = False
                        
                        for feature_key, mapping_info in feature_to_celllines.items():
                            if mapping_info['feature_name'].lower() in genetic_feature.lower():
                                if mapping_info['omic_type'] == 'cnv':
                                    cell_line_omic_data[cell_line]['cnv_features'].add(genetic_feature)
                                    found_as_cnv = True
                                elif mapping_info['omic_type'] == 'mutation':
                                    cell_line_omic_data[cell_line]['mutation_features'].add(genetic_feature)
                                    found_as_mutation = True
                        
                        # If not found in contribution analysis, use keyword-based classification
                        if not found_as_cnv and not found_as_mutation:
                            genetic_feature_lower = genetic_feature.lower()
                            if any(cnv_term in genetic_feature_lower for cnv_term in ['cnv', 'copy', 'amplification', 'deletion', 'gain', 'loss']):
                                cell_line_omic_data[cell_line]['cnv_features'].add(genetic_feature)
                            else:
                                cell_line_omic_data[cell_line]['mutation_features'].add(genetic_feature)
    
    # Create matrix DataFrame
    matrix_data = []
    
    for cell_line in all_cell_lines:
        data = cell_line_omic_data[cell_line]
        
        gene_count = len(data['gene_features'])
        protein_count = len(data['protein_features'])
        cnv_count = len(data['cnv_features'])
        mutation_count = len(data['mutation_features'])
        
        available_types = []
        if gene_count > 0:
            available_types.append('Gene')
        if protein_count > 0:
            available_types.append('Protein')
        if cnv_count > 0:
            available_types.append('CNV')
        if mutation_count > 0:
            available_types.append('Mutation')
        
        matrix_data.append({
            'Cell_Line': cell_line,
            'Gene_Count': gene_count,
            'Gene_Available': 'Yes' if gene_count > 0 else 'No',
            'Protein_Count': protein_count,
            'Protein_Available': 'Yes' if protein_count > 0 else 'No',
            'CNV_Count': cnv_count,
            'CNV_Available': 'Yes' if cnv_count > 0 else 'No',
            'Mutation_Count': mutation_count,
            'Mutation_Available': 'Yes' if mutation_count > 0 else 'No',
            'Total_Features': gene_count + protein_count + cnv_count + mutation_count,
            'Omic_Types_Available': len(available_types),
            'Data_Coverage': '; '.join(available_types) if available_types else 'None'
        })
    
    matrix_df = pd.DataFrame(matrix_data)
    
    print(f" Created Consistent matrix for {len(matrix_df)} cell lines")
    print(f"\n Omic Data Availability Summary:")
    print(f"   Gene expression: {len(matrix_df[matrix_df['Gene_Available'] == 'Yes'])} cell lines")
    print(f"   Protein expression: {len(matrix_df[matrix_df['Protein_Available'] == 'Yes'])} cell lines") 
    print(f"   CNV data: {len(matrix_df[matrix_df['CNV_Available'] == 'Yes'])} cell lines")
    print(f"   Mutation data: {len(matrix_df[matrix_df['Mutation_Available'] == 'Yes'])} cell lines")
    
    #  NEW: Show CNV matrix contribution
    cnv_from_matrix = 0
    if 'cnv_matrix' in tables and len(tables['cnv_matrix']) > 0:
        cnv_matrix_cell_lines = set(tables['cnv_matrix']['Cell_Line'].unique())
        cnv_from_matrix = len(cnv_matrix_cell_lines)
    print(f"   CNV matrix table contributed: {cnv_from_matrix} cell lines with CNV data")
    
    return matrix_df, cell_line_omic_data

def create_detailed_feature_matrix(cell_line_omic_data):
    """Create detailed feature matrix from the consistent omic data"""
    print("\n Creating detailed feature matrix...")
    
    detailed_data = []
    
    for cell_line, data in cell_line_omic_data.items():
        gene_features = list(data['gene_features'])
        protein_features = list(data['protein_features'])
        cnv_features = list(data['cnv_features'])
        mutation_features = list(data['mutation_features'])
        
        detailed_data.append({
            'Cell_Line': cell_line,
            'Gene_Features': '; '.join(gene_features[:20]) + ('...' if len(gene_features) > 20 else ''),
            'Gene_Feature_Count': len(gene_features),
            'Protein_Features': '; '.join(protein_features[:20]) + ('...' if len(protein_features) > 20 else ''),
            'Protein_Feature_Count': len(protein_features),
            'CNV_Features': '; '.join(cnv_features[:20]) + ('...' if len(cnv_features) > 20 else ''),
            'CNV_Feature_Count': len(cnv_features),
            'Mutation_Features': '; '.join(mutation_features[:20]) + ('...' if len(mutation_features) > 20 else ''),
            'Mutation_Feature_Count': len(mutation_features),
            'Total_Feature_Count': len(gene_features) + len(protein_features) + len(cnv_features) + len(mutation_features)
        })
    
    detailed_df = pd.DataFrame(detailed_data)
    print(f" Created detailed feature matrix for {len(detailed_df)} cell lines")
    
    return detailed_df

def analyze_mapping_results(enhanced_df):
    """Analyze the mapping results with special focus on CNV improvements"""
    print(f"\n Mapping Analysis Results")
    print("=" * 60)
    
    # Overall statistics
    total_features = len(enhanced_df)
    features_with_cell_lines = len(enhanced_df[enhanced_df['has_cell_line_data'] == True])
    mapping_rate = (features_with_cell_lines / total_features * 100) if total_features > 0 else 0
    
    print(f" Overall mapping rate: {mapping_rate:.1f}%")
    print(f" Features with cell line data: {features_with_cell_lines}/{total_features}")
    
    # By omic type (with special attention to CNV)
    print(f"\n Mapping rate by omic type:")
    for omic_type in enhanced_df['omic_type'].unique():
        subset = enhanced_df[enhanced_df['omic_type'] == omic_type]
        mapped = len(subset[subset['has_cell_line_data'] == True])
        total = len(subset)
        rate = (mapped / total * 100) if total > 0 else 0
        
        print(f"    {omic_type}: {rate:.1f}% ({mapped}/{total})")

def save_fixed_analysis(enhanced_df, matrix_df, detailed_df):
    """Save the FIXED analysis with CNV matrix integration"""
    print(f"\n Saving Enhanced Analysis")
    print("=" * 60)
    
    output_dir = 'mapped cell lines'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Save all files
    files_saved = []
    
    # Main enhanced analysis
    main_output = os.path.join(output_dir, 'Multi_Omic_Contribution_with_Cell_Lines_Enhanced.csv')
    enhanced_df.to_csv(main_output, index=False)
    files_saved.append(main_output)
    
    # CONSISTENT matrix with CNV
    matrix_output = os.path.join(output_dir, 'Cell_Line_Omic_Data_Matrix_with_CNV.csv')
    matrix_df.to_csv(matrix_output, index=False)
    files_saved.append(matrix_output)
    
    # Detailed features
    detailed_output = os.path.join(output_dir, 'Cell_Line_Detailed_Features_with_CNV.csv')
    detailed_df.to_csv(detailed_output, index=False)
    files_saved.append(detailed_output)
    
    # Features with mappings
    mapped_only = enhanced_df[enhanced_df['has_cell_line_data'] == True]
    mapped_output = os.path.join(output_dir, 'Features_with_Cell_Line_Mappings_Enhanced.csv')
    mapped_only.to_csv(mapped_output, index=False)
    files_saved.append(mapped_output)
    
    #  NEW: CNV-specific analysis
    cnv_only = enhanced_df[enhanced_df['omic_type'] == 'cnv']
    cnv_output = os.path.join(output_dir, 'CNV_Features_Analysis.csv')
    cnv_only.to_csv(cnv_output, index=False)
    files_saved.append(cnv_output)
    
    for file_path in files_saved:
        print(f" Saved: {file_path}")

    return output_dir

def run_enhanced_mapping(cell_line_table_dir = 'filtered_cell_line_data'):
    """Run the ENHANCED mapping analysis with CNV matrix integration"""
    
    print("Enhanced Multi-Omic Contribution Analysis + Cell Line Mapping")
    print("=" * 80)
    
    try:
        # Load data
        contrib_df = load_contribution_analysis()
        tables = load_extracted_tables(cell_line_table_dir)
        
        # Check data availability
        if len(contrib_df) == 0:
            print("\n No contribution analysis data found!")
            return None
        
        if all(len(table) == 0 for table in tables.values()):
            print("\n No extracted cell line tables found!")
            return None
        
        enhanced_df, feature_to_celllines = create_enhanced_contribution_analysis(contrib_df, tables)
        
        matrix_df, cell_line_omic_data = create_consistent_cell_line_matrix(tables, feature_to_celllines)
        
        detailed_df = create_detailed_feature_matrix(cell_line_omic_data)
        
        analyze_mapping_results(enhanced_df)
        
        output_dir = save_fixed_analysis(enhanced_df, matrix_df, detailed_df)
        
        print(f"\n Cell Line Mapping Complete!")
        print(f"\n Results saved in '{output_dir}/' directory")
        
        return enhanced_df, matrix_df, detailed_df
        
    except Exception as e:
        print(f" \nError during cell line mapping: {e}")
        import traceback
        traceback.print_exc()
        return None

In [10]:
#------------------------------------------------

In [ ]:
# import pandas as pd
# import numpy as np
# import os

# file_paths = [
#     "CCLE_DATA/CCLE_BRCA_Genetic_features_0.csv",
#     "CCLE_DATA/CCLE_BRCA_Genetic_features_1.csv", 
#     "CCLE_DATA/CCLE_RPPA_Protein.csv",
#     "CCLE_DATA/CCLE_RNAseq_genes_counts.gct",
#     "CCLE_DATA/CCLE_RNAseq_genes_rpkm.gct",
#     "CCLE_DATA/CCLE_RNAseq_rsem_genes_tpm.txt",
#     "CCLE_DATA/CCLE_cnv_matrix.txt" 
# ]

# def get_breast_cancer_cell_lines_from_data():
#     """
#     Systematically identify ALL breast cancer cell lines from the data files themselves
#     """
#     print("\n🔍 Identifying ALL breast cancer cell lines from data...")
    
#     breast_cell_lines = set()
    
#     # Method 1: From genetic features files (use GDSC tissue annotation)
#     for i, file_path in enumerate([file_paths[0], file_paths[1]]):
#         try:
#             df = pd.read_csv(file_path)
#             if 'GDSC Desc1' in df.columns:
#                 breast_lines = df[df['GDSC Desc1'] == 'breast']['Cell Line Name'].unique()
#                 breast_cell_lines.update(breast_lines)
#                 print(f"   From genetic file {i+1}: {len(breast_lines)} breast cell lines")
#         except Exception as e:
#             print(f"   Warning: Could not read genetic file {i+1}: {e}")
    
#     # Method 2: From CNV matrix (use tissue annotation)
#     try:
#         with open(file_paths[6], 'r') as f:
#             lines = f.readlines()
        
#         if len(lines) >= 2:
#             header_line = lines[0].strip().split('\t')
#             tissue_line = lines[1].strip().split('\t')
            
#             cell_line_names = header_line[3:]  # Skip first 3 columns
#             tissue_types = tissue_line[3:]     # Skip first 3 columns
            
#             cnv_breast_lines = []
#             for i, tissue in enumerate(tissue_types):
#                 if tissue == 'breast' and i < len(cell_line_names):
#                     cnv_breast_lines.append(cell_line_names[i])
            
#             breast_cell_lines.update(cnv_breast_lines)
#             print(f"   From CNV matrix: {len(cnv_breast_lines)} breast cell lines")
    
#     except Exception as e:
#         print(f"   Warning: Could not read CNV matrix: {e}")
    
#     # Method 3: From gene expression files (look for _BREAST suffix)
#     try:
#         df = pd.read_csv(file_paths[3], sep='\t', skiprows=2)
#         expression_breast_lines = [col.replace('_BREAST', '') for col in df.columns if '_BREAST' in col]
#         breast_cell_lines.update(expression_breast_lines)
#         print(f"   From expression files: {len(expression_breast_lines)} breast cell lines")
#     except Exception as e:
#         print(f"   Warning: Could not read expression files: {e}")
    
#     # Clean the cell line names
#     cleaned_breast_lines = {clean_cell_line_name(line) for line in breast_cell_lines}
#     original_breast_lines = breast_cell_lines  # Keep original names too
    
#     print(f"✅ Total unique breast cancer cell lines identified: {len(cleaned_breast_lines)}")
#     print(f"   Examples: {sorted(list(cleaned_breast_lines))[:10]}")
    
#     return cleaned_breast_lines, original_breast_lines

# def clean_cell_line_name(name):
#     """Clean cell line name by removing tissue suffixes and special characters"""
#     if '_' in name:
#         return name.split('_')[0]
#     return name.replace('-', '').replace(' ', '')

# def is_breast_cancer_cell_line(cell_line_name, known_breast_lines):
#     """
#     Check if a cell line is a breast cancer cell line using the systematically 
#     identified list instead of hard-coded keywords
#     """
#     cleaned_name = clean_cell_line_name(cell_line_name)
#     return cleaned_name in known_breast_lines

# def create_gene_symbol_mapping():
#     """Create Ensembl ID to Gene Symbol mapping from GCT file"""
#     print("\n🧬 Creating Ensembl ID → Gene Symbol mapping...")
    
#     gct_df = pd.read_csv(file_paths[3], sep='\t', skiprows=2)
    
#     gene_mapping = {}
#     for _, row in gct_df.iterrows():
#         ensembl_id = row['Name']  # ENSG00000223972.4
#         gene_symbol = row['Description']  # DDX11L1
        
#         # Store both with and without version number
#         gene_mapping[ensembl_id] = gene_symbol
#         if '.' in ensembl_id:
#             clean_id = ensembl_id.split('.')[0]
#             gene_mapping[clean_id] = gene_symbol
    
#     print(f"✅ Created mapping for {len(gene_mapping)} Ensembl IDs")
#     return gene_mapping

# def extract_table_1_genetic_features():
#     """Extract Table 1: ALL Genetic Features (Mutations) for Breast Cancer Cell Lines"""
#     print(f"\n📋 Table 1: ALL Genetic Features (Mutations) for Breast Cancer")
#     print("-" * 70)
    
#     df = pd.read_csv(file_paths[0])
#     breast_df = df[df['GDSC Desc1'] == 'breast']
#     mutated_features = breast_df[breast_df['IS Mutated'] == 1]
    
#     data_list = []
#     total_features = len(mutated_features)
    
#     for _, row in mutated_features.iterrows():
#         genetic_feature = row['Genetic Feature']
        
#         data_list.append({
#             'Cell_Line': clean_cell_line_name(row['Cell Line Name']),
#             'Cell_Line_Original': row['Cell Line Name'],
#             'COSMIC_ID': row['COSMIC ID'],
#             'Genetic_Feature': genetic_feature,
#             'IS_Mutated': row['IS Mutated'],
#             'Data_Type': 'Mutation',
#             'Source_Table': 'Table_1'
#         })
    
#     result_df = pd.DataFrame(data_list)
    
#     print(f"📊 Extracted ALL {len(result_df)} mutation features")
#     print(f"🧬 Across {result_df['Cell_Line'].nunique()} breast cancer cell lines")
#     print(f"🔬 {result_df['Genetic_Feature'].nunique()} unique genetic features")
    
#     if len(result_df) > 0:
#         print("\n📋 Sample:")
#         print(result_df.head()[['Cell_Line', 'Genetic_Feature']].to_string(index=False))
    
#     return result_df

# def extract_table_2_genetic_features():
#     """Extract Table 2: ALL Genetic Features (Mutations) for Breast Cancer Cell Lines"""
#     print(f"\n📋 Table 2: ALL Genetic Features (Mutations) for Breast Cancer")
#     print("-" * 70)
    
#     df = pd.read_csv(file_paths[1])
#     breast_df = df[df['GDSC Desc1'] == 'breast']
#     mutated_features = breast_df[breast_df['IS Mutated'] == 1]
    
#     data_list = []
#     total_features = len(mutated_features)
    
#     for _, row in mutated_features.iterrows():
#         genetic_feature = row['Genetic Feature']
        
#         data_list.append({
#             'Cell_Line': clean_cell_line_name(row['Cell Line Name']),
#             'Cell_Line_Original': row['Cell Line Name'],
#             'COSMIC_ID': row['COSMIC ID'],
#             'Genetic_Feature': genetic_feature,
#             'IS_Mutated': row['IS Mutated'],
#             'Data_Type': 'Mutation',
#             'Source_Table': 'Table_2'
#         })
    
#     result_df = pd.DataFrame(data_list)
    
#     print(f"📊 Extracted ALL {len(result_df)} mutation features")
#     print(f"🧬 Across {result_df['Cell_Line'].nunique()} breast cancer cell lines")
#     print(f"🔬 {result_df['Genetic_Feature'].nunique()} unique genetic features")
    
#     if len(result_df) > 0:
#         print("\n📋 Sample:")
#         print(result_df.head()[['Cell_Line', 'Genetic_Feature']].to_string(index=False))
    
#     return result_df

# def extract_table_3_protein_expression(known_breast_lines):
#     """Extract Table 3: ALL Protein Expression for Breast Cancer Cell Lines"""
#     print(f"\n📋 Table 3: ALL Protein Expression for Breast Cancer")
#     print("-" * 70)
    
#     df = pd.read_csv(file_paths[2])
    
#     # Use systematic breast cancer cell line identification
#     breast_df = df[df['Cell Line Name'].apply(lambda x: is_breast_cancer_cell_line(x, known_breast_lines))]
#     print(f"📊 Identified {len(breast_df)} breast cancer cell lines in protein data")
    
#     protein_cols = [col for col in df.columns if col != 'Cell Line Name']
    
#     data_list = []
#     total_features = 0
#     positive_features = 0
    
#     for _, row in breast_df.iterrows():
#         cell_line = clean_cell_line_name(row['Cell Line Name'])
        
#         for protein in protein_cols:
#             total_features += 1
#             value = row[protein]
            
#             # Include ALL protein values (positive, negative, zero)
#             if pd.notna(value):
#                 if value > 0:
#                     positive_features += 1
                    
#                 data_list.append({
#                     'Cell_Line': cell_line,
#                     'Cell_Line_Original': row['Cell Line Name'],
#                     'Protein_Name': protein,
#                     'Expression_Value': value,
#                     'Data_Type': 'Protein',
#                     'Source_Table': 'Table_3'
#                 })
    
#     result_df = pd.DataFrame(data_list)
    
#     print(f"📊 Extracted ALL {len(result_df)} protein expressions")
#     print(f"🧬 {result_df['Cell_Line'].nunique()} cell lines × {result_df['Protein_Name'].nunique()} proteins")
#     print(f"📈 {positive_features} positive expressions, {len(result_df) - positive_features} zero/negative")
#     print("🎯 Used systematic breast cancer cell line identification")
    
#     if len(result_df) > 0:
#         print("\n📋 Sample:")
#         print(result_df.head()[['Cell_Line', 'Protein_Name', 'Expression_Value']].to_string(index=False))
    
#     return result_df

# def extract_table_4_gene_expression():
#     """Extract Table 4: ALL Gene Expression Counts for Breast Cancer Cell Lines"""
#     print(f"\n📋 Table 4: ALL Gene Expression Counts for Breast Cancer")
#     print("-" * 70)
    
#     df = pd.read_csv(file_paths[3], sep='\t', skiprows=2)
#     breast_columns = [col for col in df.columns if '_BREAST' in col]
    
#     print(f"📊 Found {len(breast_columns)} breast cancer cell lines")
    
#     data_list = []
#     total_genes = len(df)
#     processed_genes = 0
    
#     for _, row in df.iterrows():
#         ensembl_id = row['Name']
#         gene_symbol = row['Description']
#         processed_genes += 1
        
#         if processed_genes % 5000 == 0:
#             print(f"   Processing gene {processed_genes}/{total_genes}...")
        
#         for col in breast_columns:
#             value = row[col]
#             if pd.notna(value) and value > 0:  # Only positive expression
#                 cell_line = clean_cell_line_name(col)
#                 data_list.append({
#                     'Cell_Line': cell_line,
#                     'Cell_Line_Original': col,
#                     'Gene_ID': ensembl_id,
#                     'Gene_Symbol': gene_symbol,
#                     'Expression_Value': value,
#                     'Data_Type': 'Gene_Counts',
#                     'Source_Table': 'Table_4'
#                 })
    
#     result_df = pd.DataFrame(data_list)
    
#     print(f"📊 Extracted ALL {len(result_df)} positive gene expressions")
#     print(f"🧬 {result_df['Cell_Line'].nunique()} cell lines × {result_df['Gene_Symbol'].nunique()} genes")
    
#     if len(result_df) > 0:
#         print("\n📋 Sample:")
#         print(result_df.head()[['Cell_Line', 'Gene_Symbol', 'Expression_Value']].to_string(index=False))
    
#     return result_df

# def extract_table_5_gene_expression():
#     """Extract Table 5: ALL Gene Expression RPKM for Breast Cancer Cell Lines"""
#     print(f"\n📋 Table 5: ALL Gene Expression RPKM for Breast Cancer")
#     print("-" * 70)
    
#     df = pd.read_csv(file_paths[4], sep='\t', skiprows=2)
#     breast_columns = [col for col in df.columns if '_BREAST' in col]
    
#     print(f"📊 Found {len(breast_columns)} breast cancer cell lines")
    
#     data_list = []
#     total_genes = len(df)
#     processed_genes = 0
    
#     for _, row in df.iterrows():
#         ensembl_id = row['Name']
#         gene_symbol = row['Description']
#         processed_genes += 1
        
#         if processed_genes % 5000 == 0:
#             print(f"   Processing gene {processed_genes}/{total_genes}...")
        
#         for col in breast_columns:
#             value = row[col]
#             if pd.notna(value) and value > 0:  # Only positive expression
#                 cell_line = clean_cell_line_name(col)
#                 data_list.append({
#                     'Cell_Line': cell_line,
#                     'Cell_Line_Original': col,
#                     'Gene_ID': ensembl_id,
#                     'Gene_Symbol': gene_symbol,
#                     'Expression_Value': value,
#                     'Data_Type': 'Gene_RPKM',
#                     'Source_Table': 'Table_5'
#                 })
    
#     result_df = pd.DataFrame(data_list)
    
#     print(f"📊 Extracted ALL {len(result_df)} positive gene expressions")
#     print(f"🧬 {result_df['Cell_Line'].nunique()} cell lines × {result_df['Gene_Symbol'].nunique()} genes")
    
#     if len(result_df) > 0:
#         print("\n📋 Sample:")
#         print(result_df.head()[['Cell_Line', 'Gene_Symbol', 'Expression_Value']].to_string(index=False))
    
#     return result_df

# def extract_table_6_gene_tpm(gene_mapping):
#     """Extract Table 6: ALL Gene TPM Expression for Breast Cancer Cell Lines"""
#     print(f"\n📋 Table 6: ALL Gene TPM Expression for Breast Cancer")
#     print("-" * 70)
    
#     # Read header to find breast cancer columns
#     with open(file_paths[5], 'r') as f:
#         header_line = f.readline().strip()
    
#     columns = header_line.split('\t')
#     if columns[0].startswith('1: '):
#         columns[0] = columns[0][3:]
    
#     breast_columns = [col for col in columns if '_BREAST' in col]
#     breast_indices = [columns.index(col) for col in breast_columns]
    
#     print(f"📊 Found {len(breast_columns)} breast cancer cell lines")
    
#     data_list = []
#     total_genes_processed = 0
    
#     with open(file_paths[5], 'r') as f:
#         header = f.readline()  # Skip header
        
#         for line_count, line in enumerate(f):
#             values = line.strip().split('\t')
#             if len(values) <= max(breast_indices):
#                 continue
                
#             total_genes_processed += 1
#             if total_genes_processed % 5000 == 0:
#                 print(f"   Processing gene {total_genes_processed}...")
                
#             gene_id = values[0]
#             transcript_ids = values[1] if len(values) > 1 else ""
            
#             # Map to gene symbol using our mapping
#             gene_symbol = gene_mapping.get(gene_id, gene_id)
#             if gene_symbol == gene_id and '.' in gene_id:
#                 clean_id = gene_id.split('.')[0]
#                 gene_symbol = gene_mapping.get(clean_id, gene_id)
            
#             # Extract values for each breast cancer cell line
#             for i, col_idx in enumerate(breast_indices):
#                 try:
#                     value = float(values[col_idx])
#                     if value > 0:  # Only positive TPM values
#                         cell_line = clean_cell_line_name(breast_columns[i])
#                         data_list.append({
#                             'Cell_Line': cell_line,
#                             'Cell_Line_Original': breast_columns[i],
#                             'Gene_ID': gene_id,
#                             'Gene_Symbol': gene_symbol,
#                             'Transcript_IDs': transcript_ids,
#                             'TPM_Value': value,
#                             'Data_Type': 'Gene_TPM',
#                             'Source_Table': 'Table_6'
#                         })
#                 except (ValueError, IndexError):
#                     continue
    
#     result_df = pd.DataFrame(data_list)
    
#     if len(result_df) > 0:
#         unique_cells = result_df['Cell_Line'].nunique()
#         unique_genes = result_df['Gene_Symbol'].nunique()
#         mapped_genes = len(result_df[result_df['Gene_Symbol'] != result_df['Gene_ID']])
        
#         print(f"📊 Extracted ALL {len(result_df)} positive TPM expressions")
#         print(f"🧬 {unique_cells} cell lines × {unique_genes} unique genes")
#         print(f"🔗 Successfully mapped {mapped_genes}/{len(result_df)} gene symbols")
        
#         print("\n📋 Sample:")
#         print(result_df.head()[['Cell_Line', 'Gene_Symbol', 'TPM_Value']].to_string(index=False))
    
#     return result_df

# def extract_table_7_cnv_features():
#     """Extract Table 7: ALL CNV Features for Breast Cancer Cell Lines"""
#     print(f"\n📋 Table 7: ALL CNV Features for Breast Cancer")
#     print("-" * 70)
    
#     with open(file_paths[6], 'r') as f:
#         lines = f.readlines()
    
#     # Parse the header structure
#     header_line = lines[0].strip().split('\t')
#     cell_line_names = header_line[3:]
    
#     tissue_line = lines[1].strip().split('\t')
#     tissue_types = tissue_line[3:]
    
#     # Find breast cancer cell lines by checking tissue type
#     breast_cell_indices = []
#     breast_cell_names = []
    
#     for i, tissue in enumerate(tissue_types):
#         if tissue == 'breast':
#             breast_cell_indices.append(i)
#             breast_cell_names.append(cell_line_names[i])
    
#     print(f"📊 Found {len(breast_cell_names)} breast cancer cell lines")
    
#     data_list = []
#     total_genes_processed = 0
#     total_cnv_values = 0
#     non_zero_cnv_values = 0
    
#     # Process each gene row
#     for line_idx in range(3, len(lines)):
#         line = lines[line_idx].strip()
#         if not line:
#             continue
            
#         values = line.split('\t')
#         if len(values) < 4:
#             continue
            
#         total_genes_processed += 1
#         if total_genes_processed % 5000 == 0:
#             print(f"   Processing gene {total_genes_processed}...")
            
#         gene_symbol = values[0] if values[0] != 'na' else None
#         gene_id = values[2] if len(values) > 2 and values[2] != 'na' else None
        
#         cnv_feature = gene_symbol if gene_symbol and gene_symbol != 'na' else gene_id
        
#         if not cnv_feature or cnv_feature == 'na':
#             continue
        
#         cnv_values = values[3:]
        
#         for i, breast_idx in enumerate(breast_cell_indices):
#             if breast_idx < len(cnv_values):
#                 try:
#                     cnv_value = float(cnv_values[breast_idx])
#                     total_cnv_values += 1
                    
#                     # Include ALL CNV values (zero, positive, negative)
#                     if cnv_value != 0.0:
#                         non_zero_cnv_values += 1
                        
#                     cell_line = clean_cell_line_name(breast_cell_names[i])
#                     data_list.append({
#                         'Cell_Line': cell_line,
#                         'Cell_Line_Original': breast_cell_names[i],
#                         'CNV_Feature': cnv_feature,
#                         'CNV_Value': cnv_value,
#                         'Data_Type': 'CNV',
#                         'Source_Table': 'Table_7'
#                     })
#                 except (ValueError, IndexError):
#                     continue
    
#     result_df = pd.DataFrame(data_list)
    
#     print(f"📊 Extracted ALL {len(result_df)} CNV mappings")
#     print(f"🧬 {result_df['Cell_Line'].nunique()} cell lines × {result_df['CNV_Feature'].nunique()} CNV features")
#     print(f"📈 {non_zero_cnv_values} non-zero CNVs, {total_cnv_values - non_zero_cnv_values} normal copies")
    
#     if len(result_df) > 0:
#         print("\n📋 Sample:")
#         print(result_df.head()[['Cell_Line', 'CNV_Feature', 'CNV_Value']].to_string(index=False))
    
#     return result_df

# def save_all_results(results_dict, breast_lines_info):
#     """Save all complete results to organized CSV files"""
#     print(f"\n💾 Saving Complete Multi-Omics Tables")
#     print("=" * 60)
    
#     output_dir = 'complete_breast_multiomics_all_features'
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
    
#     saved_files = []
#     summary_data = []
    
#     for table_name, df in results_dict.items():
#         if len(df) > 0:
#             output_path = os.path.join(output_dir, f"{table_name}.csv")
#             df.to_csv(output_path, index=False)
#             saved_files.append(output_path)
            
#             unique_cells = df['Cell_Line'].nunique()
            
#             if 'Genetic_Feature' in df.columns:
#                 unique_features = df['Genetic_Feature'].nunique()
#                 feature_type = 'Genetic Features'
#             elif 'Protein_Name' in df.columns:
#                 unique_features = df['Protein_Name'].nunique()
#                 feature_type = 'Proteins'
#             elif 'Gene_Symbol' in df.columns:
#                 unique_features = df['Gene_Symbol'].nunique()
#                 feature_type = 'Genes'
#             elif 'CNV_Feature' in df.columns:
#                 unique_features = df['CNV_Feature'].nunique()
#                 feature_type = 'CNV Features'
#             else:
#                 unique_features = 0
#                 feature_type = 'Unknown'
            
#             summary_data.append({
#                 'Table': table_name,
#                 'Feature_Type': feature_type,
#                 'Total_Mappings': len(df),
#                 'Unique_Cell_Lines': unique_cells,
#                 'Unique_Features': unique_features,
#                 'Source': df['Source_Table'].iloc[0] if 'Source_Table' in df.columns else 'Unknown'
#             })
            
#             print(f"✅ Saved: {output_path} ({len(df):,} mappings)")
    
#     # Save summary report
#     summary_df = pd.DataFrame(summary_data)
#     summary_path = os.path.join(output_dir, 'Complete_Summary_Report.csv')
#     summary_df.to_csv(summary_path, index=False)
#     print(f"✅ Saved: {summary_path}")
    
#     # Save systematically identified breast cancer cell lines
#     cleaned_lines, original_lines = breast_lines_info
#     breast_lines_df = pd.DataFrame({
#         'Breast_Cancer_Cell_Lines_Cleaned': sorted(list(cleaned_lines)),
#     })
    
#     # Add original names
#     original_df = pd.DataFrame({
#         'Breast_Cancer_Cell_Lines_Original': sorted(list(original_lines))
#     })
    
#     breast_lines_path = os.path.join(output_dir, 'All_Breast_Cancer_Cell_Lines.csv')
#     breast_lines_df.to_csv(breast_lines_path, index=False)
#     print(f"✅ Saved: {breast_lines_path}")
    
#     original_lines_path = os.path.join(output_dir, 'All_Breast_Cancer_Cell_Lines_Original.csv')
#     original_df.to_csv(original_lines_path, index=False)
#     print(f"✅ Saved: {original_lines_path}")
    
#     # Create master cell line list from results
#     all_cell_lines = set()
#     for df in results_dict.values():
#         if len(df) > 0 and 'Cell_Line' in df.columns:
#             all_cell_lines.update(df['Cell_Line'].unique())
    
#     cell_line_df = pd.DataFrame({
#         'Cell_Lines_With_Multi_Omic_Data': sorted(list(all_cell_lines))
#     })
#     cell_line_path = os.path.join(output_dir, 'Cell_Lines_With_Data.csv')
#     cell_line_df.to_csv(cell_line_path, index=False)
#     print(f"✅ Saved: {cell_line_path}")
    
#     print(f"\n📊 COMPLETE EXTRACTION SUMMARY:")
#     total_mappings = sum(len(df) for df in results_dict.values())
#     print(f"   📈 Total multi-omic mappings: {total_mappings:,}")
#     print(f"   🧬 Unique breast cancer cell lines: {len(all_cell_lines)}")
#     print(f"   🎯 Systematic identification (no hard-coded keywords)")
#     print(f"   📁 All files saved in '{output_dir}/' directory")

# def run_complete_extraction():
#     """Run complete multi-omics extraction for ALL breast cancer features"""
#     print("🚀 COMPLETE BREAST CANCER MULTI-OMICS EXTRACTION")
#     print("=" * 80)
#     print("Extracting ALL multi-omic features for ALL breast cancer cell lines")
#     print("=" * 80)
    
#     # Systematically identify breast cancer cell lines
#     breast_lines_info = get_breast_cancer_cell_lines_from_data()
#     cleaned_lines, original_lines = breast_lines_info
    
#     if not cleaned_lines:
#         print("❌ No breast cancer cell lines identified!")
#         return None
    
#     # Create gene mapping
#     gene_mapping = create_gene_symbol_mapping()
    
#     results = {}
    
#     try:
#         results['Table_1_All_Genetic_Features'] = extract_table_1_genetic_features()
#         results['Table_2_All_Genetic_Features'] = extract_table_2_genetic_features()
#         results['Table_3_All_Protein_Expression'] = extract_table_3_protein_expression(cleaned_lines)
#         results['Table_4_All_Gene_Expression_Counts'] = extract_table_4_gene_expression()
#         results['Table_5_All_Gene_Expression_RPKM'] = extract_table_5_gene_expression()
#         results['Table_6_All_Gene_TPM_Expression'] = extract_table_6_gene_tpm(gene_mapping)
#         results['Table_7_All_CNV_Features'] = extract_table_7_cnv_features()
        
#         save_all_results(results, breast_lines_info)
        
#         print(f"\n✅ COMPLETE EXTRACTION FINISHED!")
#         print(f"📊 All multi-omic features for all breast cancer cell lines extracted")
#         print(f"🎯 Systematic breast cancer identification used")
#         print(f"🧬 Ready for comprehensive multi-omic analysis!")
        
#         return results
        
#     except Exception as e:
#         print(f"❌ Error during extraction: {e}")
#         import traceback
#         traceback.print_exc()
#         return None